In [ ]:
# 3 - modeling
# The purpose of this notebook is to create an onnx file for the best model. This model is the same one found in
# 3.13-final-model.ipynb

In [2]:
# with the old onnx file, lets see if we get the same predictions as arron

#
# now lets do inference with an image using the ONNX model
#
import cv2
import numpy as np
import onnxruntime as ort

image_path = r"C:\Users\bxb370\GelSiteMiniFlowAndLevelingModel\data_test\{leveling-2.5}_P{Brooke}_D{7.21.26}_type{TEST}_GS{2BDR-9F02}_Panel{GS-25-103}_0degrees_flat.png"

# Load raw grayscale image
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError(image_path)

print("Image shape:", img.shape)  # should be (2464, 3280)

# Convert to float32
img = img.astype(np.float32)

# Add batch and channel dimensions
img = img[np.newaxis, np.newaxis, :, :]

print("Input shape:", img.shape)  # (1, 1, 2464, 3280)

# Load ONNX model
session = ort.InferenceSession("waveletcnn_full.onnx")

# Run inference
pred = session.run(
    None,
    {"image": img}
)

score = float(pred[0][0][0])

print(f"Predicted Leveling Score: {score:.3f}")

Image shape: (2464, 3280)
Input shape: (1, 1, 2464, 3280)
Predicted Leveling Score: 3.136


In [3]:
#
# insight: the onnx file is getting the same prediction as arron is, we will need to create a new file
# for the updated model to include the updated weights resulting from adding the new data (data_new_2)
#

In [4]:
#
# ONNX file for new best model
#

In [2]:
# imports

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

import sys
import os
import importlib
import numpy as np

sys.path.append(os.path.abspath(".."))
from src.data import load_images_from_metadata, filter_images

In [6]:
# load all of the data into metadata csv file including the new data (data_new_2))
from src.build_metadata import load_metadata
import importlib
import src.build_metadata as build_metadata

importlib.reload(build_metadata)
build_metadata.load_metadata(
    data_base_dir="../data",
    new_data_base_dir="../data_new",
    new_data_2_base_dir="../data_new_2",
    output_file="metadata.csv",
)

Applied human ratings to 851 DD rows.
Saved metadata to ../data\metadata.csv
Combined records: 1345


,LevelingScore,Person,DateCollected,ImageType,GSCamera,PanelID,State,FilePath
0,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (1...
1,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (2...
2,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (3...
3,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (4...
4,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (5...
...,...,...,...,...,...,...,...,...
1340,5.8,Brooke,7.21.26,TEST,2BDR-9F02,25-143,flat,../data_test\{leveling-5.8}_P{Brooke}_D{7.21.2...
1341,6.3,Brooke,7.21.26,TEST,2BDR-9F02,25-141,flat,../data_test\{leveling-6.3}_P{Brooke}_D{7.21.2...
1342,6.8,Brooke,7.21.26,TEST,2BDR-9F02,25-104,flat,../data_test\{leveling-6.8}_P{Brooke}_D{7.21.2...
1343,8.0,Brooke,7.21.26,TEST,2BDR-9F02,25-125,flat,../data_test\{leveling-8.0}_P{Brooke}_D{7.21.2...


In [7]:
# get the data
data = load_images_from_metadata("../data/metadata.csv", crop_fraction = .4, use_cv2=True)

data_standards = filter_images(data, ImageType = "STD", State = "flat", DateCollected = ["6.2.2026", "6.3.2026", "6.5.2026", "6.8.2026"])
data_real_paint = filter_images(data, ImageType = "DD", State = "flat")

Loaded 1345 images (cropping=ON, backend=cv2)
Filtered down to 177 images
Filtered down to 430 images


In [8]:
# nessary classes/ functions

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn as nn


def prepare_tensors(data, img_size=(64, 64)):
    X, y, groups = [], [], []
    for d in data:
        img = d["image"].astype(np.float32) / 255.0
        if img.ndim != 2:
            raise ValueError(f"Expected grayscale 2D image, got shape {img.shape}")

        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        img = torch.nn.functional.interpolate(
            img.unsqueeze(0),
            size=img_size,
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        X.append(img)
        y.append(float(d["LevelingScore"]))
        groups.append(d.get("PanelID", None))

    return X, np.array(y, dtype=np.float32), np.array(groups)

def make_transforms(img_size, rotation_deg, crop_scale):
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(rotation_deg),
        transforms.RandomResizedCrop(
            img_size,
            scale=(crop_scale, 1.0),
            antialias=True,
        ),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    val_tf = transforms.Compose([
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    return train_tf, val_tf

class TensorListDataset(Dataset):
    def __init__(self, X_list, y_array, transform=None):
        self.X_list = X_list
        self.y_array = y_array
        self.transform = transform

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        x = self.X_list[idx].clone()
        if self.transform is not None:
            x = self.transform(x)
        y = torch.tensor(self.y_array[idx], dtype=torch.float32)
        return x, y

def weighted_sampler_from_y(y, seed=None):
    y_int = np.rint(y).astype(int)
    unique, counts = np.unique(y_int, return_counts=True)
    freq = {k: c for k, c in zip(unique, counts)}
    weights = np.array([1.0 / freq[v] for v in y_int], dtype=np.float32)

    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)

    return WeightedRandomSampler(
        torch.tensor(weights),
        num_samples=len(weights),
        replacement=True,
        generator=generator,
    )

class WaveletCNNRegressor(nn.Module):
    def __init__(self, num_blocks=5, base_channels=16, hidden_dim=128, dropout=0.5, use_batchnorm=True):
        super().__init__()
        self.pool = nn.AvgPool2d(2)

        layers = []
        in_channels = 1
        channels = base_channels
        for i in range(num_blocks):
            layers.append(nn.Conv2d(in_channels, channels, 3, padding=1))
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(channels))
            layers.extend([
                nn.ReLU(),
                nn.Conv2d(channels, channels, 3, padding=1),
                nn.ReLU(),
            ])
            if i < num_blocks - 1:
                layers.append(nn.MaxPool2d(2))

            in_channels = channels
            channels *= 2

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        self.features = nn.Sequential(*layers)

        self.regressor = nn.Sequential(
            nn.Linear(in_channels + 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        x_low1 = self.pool(x)
        x_low2 = self.pool(x_low1)

        m0 = x.mean(dim=[2, 3])
        m1 = x_low1.mean(dim=[2, 3])
        m2 = x_low2.mean(dim=[2, 3])
        wavelet_feats = torch.cat([m0, m1, m2], dim=1)

        feats = self.features(x)
        feats = feats.view(feats.size(0), -1)

        combined = torch.cat([feats, wavelet_feats], dim=1)
        return self.regressor(combined).squeeze(1)

def make_optimizer(name, params, lr, weight_decay):
    if name == "Adam":
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "RMSprop":
        return optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)

def make_loss(name):
    if name == "L1":
        return nn.L1Loss()
    if name == "MSE":
        return nn.MSELoss()
    return nn.SmoothL1Loss(beta=0.5)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
# model with fixed wavelet parameters: train on standards, evaluate on real paint, save final weights each run

import os
import random
from datetime import datetime
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader

# Ensure required objects from earlier cells exist
required_names = [
    "data_standards",
    "data_real_paint",
    "prepare_tensors",
    "make_transforms",
    "TensorListDataset",
    "weighted_sampler_from_y",
    "WaveletCNNRegressor",
    "make_optimizer",
    "make_loss",
    "DEVICE",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Run earlier setup cells first. Missing: {missing}")

# Randomize initialization every run (different seed each execution)
run_seed = 3
random.seed(run_seed)
np.random.seed(run_seed)
torch.manual_seed(run_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(run_seed)

# Fixed parameters you provided
fixed_p = {
    "img_size": 92,
    "num_blocks": 5,
    "base_channels": 12,
    "hidden_dim": 256,
    "dropout": 0.2511572371061875,
    "use_batchnorm": True,
    "batch_size": 32,
    "optimizer": "Adam",
    "lr": 0.0006049716507542575,
    "weight_decay": 1.963434157293331e-08,
    "rotation_deg": 6,
    "crop_scale": 0.9608106745617722,
    "loss": "SmoothL1",
}
epochs = 70

img_size = int(fixed_p["img_size"])
X_std, y_std, groups_std = prepare_tensors(data_standards, img_size=(img_size, img_size))
X_real, y_real, groups_real = prepare_tensors(data_real_paint, img_size=(img_size, img_size))

train_tf, test_tf = make_transforms(
    img_size=img_size,
    rotation_deg=int(fixed_p["rotation_deg"]),
    crop_scale=float(fixed_p["crop_scale"]),
)

# Train on all standards
std_train_ds = TensorListDataset(X_std, y_std, transform=train_tf)
std_sampler = weighted_sampler_from_y(y_std, seed=run_seed)
std_train_loader = DataLoader(
    std_train_ds,
    batch_size=int(fixed_p["batch_size"]),
    sampler=std_sampler,
    drop_last=False,
)

# Evaluate on all real paint
real_test_ds = TensorListDataset(X_real, y_real, transform=test_tf)
real_test_loader = DataLoader(
    real_test_ds,
    batch_size=int(fixed_p["batch_size"]),
    shuffle=False,
    drop_last=False,
)

model_fixed = WaveletCNNRegressor(
    num_blocks=int(fixed_p["num_blocks"]),
    base_channels=int(fixed_p["base_channels"]),
    hidden_dim=int(fixed_p["hidden_dim"]),
    dropout=float(fixed_p["dropout"]),
    use_batchnorm=bool(fixed_p["use_batchnorm"]),
).to(DEVICE)

optimizer_fixed = make_optimizer(
    fixed_p["optimizer"],
    model_fixed.parameters(),
    float(fixed_p["lr"]),
    float(fixed_p["weight_decay"]),
)
criterion_fixed = make_loss(fixed_p["loss"])

print("=" * 76)
print("FIXED-PARAM WAVELET MODEL | TRAIN: STANDARDS (70 EPOCHS) | TEST: REAL PAINT")
print("Random initialization enabled: new seed per run")
print("=" * 76)
print(f"Run seed: {run_seed}")
print(f"Standards train size: {len(X_std)} | Real-paint test size: {len(X_real)}")

for epoch in range(epochs):
    model_fixed.train()
    running_loss = 0.0
    n_seen = 0

    for images, labels in std_train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_fixed.zero_grad()
        preds = model_fixed(images)
        loss = criterion_fixed(preds, labels)
        loss.backward()
        optimizer_fixed.step()

        batch_n = images.size(0)
        running_loss += loss.item() * batch_n
        n_seen += batch_n

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch + 1}/{epochs} - train loss: {running_loss / max(1, n_seen):.4f}")

# Inference on real paint
model_fixed.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for images, labels in real_test_loader:
        images = images.to(DEVICE)
        preds = model_fixed(images)
        preds_all.append(preds.detach().cpu().numpy())
        labels_all.append(labels.numpy())

preds_np = np.concatenate(preds_all)
labels_np = np.concatenate(labels_all)
abs_err = np.abs(preds_np - labels_np)

print("\nReal-paint evaluation (raw, no calibration):")
print(f"  MAE: {float(abs_err.mean()):.4f}")
print(f"  Within +/-1: {float((abs_err <= 1).mean() * 100):.2f}%")
print(f"  Within +/-2: {float((abs_err <= 2).mean() * 100):.2f}%")
print(f"  Within +/-3: {float((abs_err <= 3).mean() * 100):.2f}%")
print(f"  Within +/-4: {float((abs_err <= 4).mean() * 100):.2f}%")

results_real_fixed_df = pd.DataFrame(
    {
        "PanelID": groups_real,
        "Prediction": preds_np,
        "TrueLabel": labels_np,
    }
)
results_real_fixed_df["Difference"] = results_real_fixed_df["Prediction"] - results_real_fixed_df["TrueLabel"]
results_real_fixed_df["AbsDifference"] = np.abs(results_real_fixed_df["Difference"])

print("\nPreview:")
display(results_real_fixed_df.head(20))

# Save final model weights and run metadata
os.makedirs("../models", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"../models/wavelet_fixed_standards_to_real_e70_seed{run_seed}_{timestamp}.pt"

checkpoint = {
    "model_state_dict": model_fixed.state_dict(),
    "params": fixed_p,
    "epochs": epochs,
    "run_seed": run_seed,
    "metrics": {
        "mae": float(abs_err.mean()),
        "within_1": float((abs_err <= 1).mean() * 100),
        "within_2": float((abs_err <= 2).mean() * 100),
        "within_3": float((abs_err <= 3).mean() * 100),
        "within_4": float((abs_err <= 4).mean() * 100),
    },
}
torch.save(checkpoint, save_path)
print(f"\nSaved final model checkpoint to: {save_path}")

FIXED-PARAM WAVELET MODEL | TRAIN: STANDARDS (70 EPOCHS) | TEST: REAL PAINT
Random initialization enabled: new seed per run
Run seed: 3
Standards train size: 177 | Real-paint test size: 430
  Epoch 1/70 - train loss: 3.8307
  Epoch 10/70 - train loss: 0.7351
  Epoch 20/70 - train loss: 0.5065
  Epoch 30/70 - train loss: 0.4607
  Epoch 40/70 - train loss: 0.6680
  Epoch 50/70 - train loss: 0.5637
  Epoch 60/70 - train loss: 0.4271
  Epoch 70/70 - train loss: 0.5344

Real-paint evaluation (raw, no calibration):
  MAE: 0.6798
  Within +/-1: 79.07%
  Within +/-2: 96.74%
  Within +/-3: 99.30%
  Within +/-4: 100.00%

Preview:


,PanelID,Prediction,TrueLabel,Difference,AbsDifference
0,25-130,1.856636,1.3,0.556636,0.556636
1,25-130,1.732070,1.3,0.432070,0.432070
2,25-130,1.407076,1.3,0.107076,0.107076
3,25-130,1.696472,1.3,0.396472,0.396472
4,25-130,1.975164,1.3,0.675164,0.675164
5,25-130,1.679193,1.3,0.379193,0.379193
6,25-101,3.155988,2.2,0.955988,0.955988
7,25-101,2.841805,2.2,0.641805,0.641805
8,25-101,2.895194,2.2,0.695194,0.695194
9,25-101,2.765134,2.2,0.565134,0.565134



Saved final model checkpoint to: ../models/wavelet_fixed_standards_to_real_e70_seed3_20260731_113139.pt


In [10]:
# onnx file

In [11]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
import numpy as np
import cv2


class FullInferenceModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # Input: [N, 1, 2464, 3280], float32, pixel values 0..255
        # Crop 20% from each side (crop_fraction=0.4)
        x = x[:, :, 492:1972, 656:2624]
        
        # Resize to match training size (92x92)
        img_size = int(fixed_p["img_size"])
        x = F.interpolate(x, size=(img_size, img_size), mode="bilinear", align_corners=False)
        
        # Scale to [0,1] then normalize(mean=0.5, std=0.5)
        x = x / 255.0
        x = (x - 0.5) / 0.5
        
        return self.model(x)


model_fixed.eval()

full_model = FullInferenceModel(model_fixed)
full_model.eval()

# Dummy input matches raw image dimensions
dummy_input = torch.zeros(1, 1, 2464, 3280, dtype=torch.float32)

# Absolute path avoids mixed-separator issues on Windows
save_path = os.path.abspath(os.path.join("..", "models", "waveletcnn_v2.onnx"))

torch.onnx.export(
    full_model,
    dummy_input,
    save_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["image"],
    output_names=["score"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "score": {0: "batch_size"},
    },
)

print(f"Exported to {save_path}")


C:\Users\bxb370\AppData\Local\Temp\ipykernel_22860\925321769.py:42: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0731 11:31:40.862000 22860 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `FullInferenceModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `FullInferenceModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


c:\Users\bxb370\AppData\Local\Programs\Python\Python314\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\bxb370\AppData\Local\Programs\Python\Python314\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\bxb370\AppData\Local\Programs\Python\Python314\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Use

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported to c:\Users\bxb370\GelSiteMiniFlowAndLevelingModel\models\waveletcnn_v2.onnx


In [ ]:
#
# Test cases
#

In [8]:
# Test ONNX inference on a real image # 1

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
import numpy as np
import cv2

save_path = os.path.abspath(os.path.join("..", "models", "waveletcnn_v2.onnx"))

image_path = r"C:\Users\bxb370\GelSiteMiniFlowAndLevelingModel\data_test\{leveling-2.5}_P{Brooke}_D{7.21.26}_type{TEST}_GS{2BDR-9F02}_Panel{GS-25-103}_0degrees_flat.png"

# Load raw grayscale image
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError(image_path)

print("Image shape:", img.shape)  # should be (2464, 3280)

# Convert to float32
img = img.astype(np.float32)

# Add batch and channel dimensions
img = img[np.newaxis, np.newaxis, :, :]

print("Input shape:", img.shape)  # (1, 1, 2464, 3280)

# Load ONNX model
session = ort.InferenceSession(save_path)

# Run inference
onnx_raw = session.run(
    None,
    {"image": img}
)

# WaveletCNNRegressor returns shape (batch,), so extract first element
score = float(onnx_raw[0].flat[0])

print(f"Predicted Leveling Score: {score:.3f}")


Image shape: (2464, 3280)
Input shape: (1, 1, 2464, 3280)
Predicted Leveling Score: 3.079


In [9]:
# Test ONNX inference on a real image # 2

image_path = r"C:\Users\bxb370\GelSiteMiniFlowAndLevelingModel\data_test\{leveling-5.8}_P{Brooke}_D{7.21.26}_type{TEST}_GS{2BDR-9F02}_Panel{GS-25-143}_0degrees_flat.png"

# Load raw grayscale image
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError(image_path)

print("Image shape:", img.shape)  # should be (2464, 3280)

# Convert to float32
img = img.astype(np.float32)

# Add batch and channel dimensions
img = img[np.newaxis, np.newaxis, :, :]

print("Input shape:", img.shape)  # (1, 1, 2464, 3280)

# Load ONNX model
session = ort.InferenceSession(save_path)

# Run inference
onnx_raw = session.run(
    None,
    {"image": img}
)

# WaveletCNNRegressor returns shape (batch,), so extract first element
score = float(onnx_raw[0].flat[0])

print(f"Predicted Leveling Score: {score:.3f}")


Image shape: (2464, 3280)
Input shape: (1, 1, 2464, 3280)
Predicted Leveling Score: 5.502


In [7]:
# Test ONNX inference on a real image # 3

image_path = r"C:\Users\bxb370\GelSiteMiniFlowAndLevelingModel\data_test\{leveling-6.8}_P{Brooke}_D{7.21.26}_type{TEST}_GS{2BDR-9F02}_Panel{GS-25-104}_0degrees_flat.png"

# Load raw grayscale image
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError(image_path)

print("Image shape:", img.shape)  # should be (2464, 3280)

# Convert to float32
img = img.astype(np.float32)

# Add batch and channel dimensions
img = img[np.newaxis, np.newaxis, :, :]

print("Input shape:", img.shape)  # (1, 1, 2464, 3280)

# Load ONNX model
session = ort.InferenceSession(save_path)

# Run inference
onnx_raw = session.run(
    None,
    {"image": img}
)

# WaveletCNNRegressor returns shape (batch,), so extract first element
score = float(onnx_raw[0].flat[0])

print(f"Predicted Leveling Score: {score:.3f}")


Image shape: (2464, 3280)
Input shape: (1, 1, 2464, 3280)
Predicted Leveling Score: 6.240
